In [27]:
# pip install transformers accelerate torch --upgrade
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from functools import reduce

In [2]:
bdf = pd.read_csv("./all_codes_full_wide_aug5.csv")
f_poli_leaning_map = {
    -6: "FAR_LEFT",
    -5: "FAR_LEFT",
    -4: "FAR_LEFT",
    -3: "FAR_LEFT",
    -2: "LEFT",
    -1: "LEAN_LEFT",
    0: "CENTER",
    1: "LEAN_RIGHT",
    2: "RIGHT",
    3: "FAR_RIGHT",
    4: "FAR_RIGHT",
    5: "FAR_RIGHT",
    6: "FAR_RIGHT",
    None: "UNDEFINED",
    42: "UNDEFINED"
}
fdf = bdf[bdf.columns[~bdf.columns.str.endswith("emmet")]]
fdf["sodi_poli_leaning_5"] = fdf["user_poli_leaning_num_sodi"].map(f_poli_leaning_map)

/scratch/slurm-1441438/ipykernel_368696/1451926091.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fdf["sodi_poli_leaning_5"] = fdf["user_poli_leaning_num_sodi"].map(f_poli_leaning_map)


In [34]:
# fdf = fdf.head()
# fdf = fdf[~fdf['sodi_aireg_stance'].isin(["NEUTRAL", "UNDEFINED"])]

In [40]:
# fdf[~fdf['sodi_aireg_stance'].isin(["NEUTRAL", "UNDEFINED"])]['sodi_aireg_stance'].shape

# fdf['sodi_aireg_stance'].value_counts()
fdf.columns

Index(['user_poli_leaning_num_sodi', 'user_agree_with_sodi',
       'user_article_subject_sodi', 'user_wants_to_see_num_sodi',
       'user_ai_leaning_num_sodi', 'user_aireg_leaning_num_sodi',
       'user_imm_leaning_num_sodi', 'user_bias_amnt_num_sodi',
       'user_bias_cause_sodi', 'coding_date_sodi', 'coding_time_sodi',
       'gpt_leaning', 'pull', 'title', 'text', 'date', 'gkg_id', 'set_number',
       'source_url', 'coded', 'article_set_unique_id', 'human_clean_text',
       'human_clean_title', 'source', 'source_leaning', 'allsides_leaning',
       'claude_gpt_summary', 'claude_gpt_poli_leaning',
       'claude_gpt_poli_justification', 'claude_gpt_article_subject',
       'claude_gpt_ai_leaning', 'claude_gpt_ai_justification',
       'claude_gpt_aireg_leaning', 'claude_gpt_aireg_justification',
       'claude_gpt_imm_justification', 'claude_gpt_primary_bias_amnt',
       'claude_gpt_primary_bias_cause', 'claude_gpt_imm_leaning',
       'gpt4o_gpt_summary', 'gpt4o_gpt_poli_lean

In [22]:
TEXT_COL = "text"
MODEL_NAME= "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8

# Hypothesis sets for each stance polarity (paraphrased for ensembling)
# PROREG_HYPOTHESES = [
#     "The author supports stronger government regulation of AI.",
#     "The author argues that AI should be more tightly regulated.",
#     "The text is in favor of increasing AI regulation."
# ]
# ANTIREG_HYPOTHESES = [
#     "The author opposes government regulation of AI.",
#     "The author argues that AI should be less regulated.",
#     "The text is against increasing AI regulation."
# ]

# PROAI_HYPOTHESES = [
#     "The author is optimistic about AI.",
#     "The author believes that AI is promising and should be developed further.",
#     "The text is in favor of AI."
# ]
# ANTIAI_HYPOTHESES = [
#     "The author is pessimistic about AI.",
#     "The author argues that AI is not desirable and should not be developed further.",
#     "The text is against AI."
# ]


# PRO_HYPOTHESES = [
#     "The author is permissive about immigration.",
#     "The author believes immigrants should be welcome in their country.",
#     "The text is in favor of immigration."
# ]
# ANTI_HYPOTHESES = [
#     "The author is strict about immigration.",
#     "The author believes that immigration is a problem and should be hampered.",
#     "The text is against immigration."
# ]

PRO_HYPOTHESES = [
    "The author is politically progressive.",
    "The author believes that the government should do more for it's citizens.",
    "The text is in line with a liberal perspective.",
    "The text expresses a progressive viewpoint.",
    "The text presents arguments in line with a liberal perspective.",
    "The text portrays liberal positions positively."
    
]
ANTI_HYPOTHESES = [
    "The author is politically conservative.",
    "The author believes that the government should be curtailed and let the private sector do more.",
    "The text is in line with a conservative agenda.",
    "The text portrays conservative positions positively",
    "The text expresses a conservative viewpoint."
]
# Decision thresholds:
# - if max entailment < MIN_SUPPORT -> NEUTRAL
# - else pick argmax between PRO vs ANTI if the winning side exceeds the other by MARGIN
MIN_SUPPORT = 0.35
MARGIN = 0.05

In [23]:

# Helpers
def nli_entail_probs(premises, hypotheses):
    """
    Return entailment probabilities for each (premise, hypothesis) pair.
    Shape: (len(premises),)
    """
    with torch.no_grad():
        inputs = tokenizer(premises, hypotheses, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        logits = model(**inputs).logits  # label order typically: contradiction, neutral, entailment
        probs = torch.softmax(logits, dim=-1)
        entail = probs[:, -1]  # entailment column
    return entail.detach().cpu().numpy()

def stance_scores(texts):
    """
    For a list of texts, compute averaged entailment scores for PRO and ANTI
    across the hypothesis paraphrases.
    Returns (pro_scores, anti_scores), each np.array of shape (N,)
    """
    # average over hypotheses
    pro_all = []
    for h in PRO_HYPOTHESES:
        pro_all.append(nli_entail_probs(texts, [h]*len(texts)))
    anti_all = []
    for h in ANTI_HYPOTHESES:
        anti_all.append(nli_entail_probs(texts, [h]*len(texts)))

    pro_mean = np.vstack(pro_all).mean(axis=0)
    anti_mean = np.vstack(anti_all).mean(axis=0)
    return pro_mean, anti_mean

def decide_labels(pro, anti, min_support=MIN_SUPPORT, margin=MARGIN):

    labels = []
    confs = []
    for p, a in zip(pro, anti):
        mx = max(p, a)
        if mx < min_support:
            labels.append("UNDEFINED")
            confs.append(1 - mx)  # low confidence when both weak
            continue
        if p - a > margin:
            # labels.append("PROREG")
            # labels.append("OPTIMISTIC")
            # labels.append("PERMISSIVE")
            labels.append("LEFT")
            confs.append(p)
        elif a - p > margin:
            # labels.append("ANTIREG")
            # labels.append("PESSIMISTIC")
            # labels.append("STRICT")
            labels.append("RIGHT")
            confs.append(a)
        else:
            labels.append("NEUTRAL")
            confs.append(1 - abs(p - a))
    return np.array(labels), np.array(confs)

In [ ]:
# ---- Load model ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()




In [25]:

# ---- Apply to your DataFrame fdf ----
assert TEXT_COL in fdf.columns, f"'{TEXT_COL}' column not found in fdf."
texts = fdf[TEXT_COL].fillna("").astype(str).tolist()

labels_out = []
pro_out = []
anti_out = []
conf_out = []

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    pro, anti = stance_scores(batch)
    labels, confs = decide_labels(pro, anti)
    labels_out.extend(labels)
    pro_out.extend(pro)
    anti_out.extend(anti)
    conf_out.extend(confs)

# fdf["moritz_aireg_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
# fdf["moritz_aireg_pro_score"] = pro_out            # entailment prob for pro-reg
# fdf["moritz_aireg_anti_score"] = anti_out          # entailment prob for anti-reg
# fdf["moritz_aireg_confidence"] = conf_out          # crude confidence metric

# fdf["moritz_ai_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
# fdf["moritz_ai_pro_score"] = pro_out            # entailment prob for pro-reg
# fdf["moritz_ai_anti_score"] = anti_out          # entailment prob for anti-reg
# fdf["moritz_ai_confidence"] = conf_out          # crude confidence metric


# fdf["moritz_imm_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
# fdf["moritz_imm_pro_score"] = pro_out            # entailment prob for pro-reg
# fdf["moritz_imm_anti_score"] = anti_out          # entailment prob for anti-reg
# fdf["moritz_imm_confidence"] = conf_out          # crude confidence metric

fdf["moritz_poli_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
fdf["moritz_poli_pro_score"] = pro_out            # entailment prob for pro-reg
fdf["moritz_poli_anti_score"] = anti_out          # entailment prob for anti-reg
fdf["moritz_poli_confidence"] = conf_out          # crude confidence metric


# Quick sanity check
# aireg_final_mortiz_df = fdf[["article_set_unique_id", "moritz_aireg_stance","moritz_aireg_pro_score","moritz_aireg_anti_score","moritz_aireg_confidence"]]
# aireg_final_mortiz_df.to_csv("aireg_final_mortiz_df.csv")
# ai_final_mortiz_df = fdf[["article_set_unique_id", "moritz_ai_stance","moritz_ai_pro_score","moritz_ai_anti_score","moritz_ai_confidence"]]
# ai_final_mortiz_df.to_csv("ai_final_mortiz_df.csv")
# ai_final_mortiz_df = fdf[["article_set_unique_id", "moritz_imm_stance","moritz_imm_pro_score","moritz_imm_anti_score","moritz_imm_confidence"]]
# ai_final_mortiz_df.to_csv("imm_final_mortiz_df.csv")
ai_final_mortiz_df = fdf[["article_set_unique_id", "moritz_poli_stance","moritz_poli_pro_score","moritz_poli_anti_score","moritz_poli_confidence"]]
ai_final_mortiz_df.to_csv("poli_final_mortiz_df.csv")

  0%|          | 0/54 [00:00<?, ?it/s]

/scratch/slurm-1441438/ipykernel_368696/2416528636.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fdf["moritz_poli_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
/scratch/slurm-1441438/ipykernel_368696/2416528636.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fdf["moritz_poli_pro_score"] = pro_out            # entailment prob for pro-reg


In [14]:
# len(labels_out)
labels_out = [x for x in labels_out if x != "ANTIREG"]
fdf["moritz_ai_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
fdf["moritz_ai_pro_score"] = pro_out            # entailment prob for pro-reg
fdf["moritz_ai_anti_score"] = anti_out          # entailment prob for anti-reg
fdf["moritz_ai_confidence"] = conf_out          # crude confidence metric

# Quick sanity check
# aireg_final_mortiz_df = fdf[["article_set_unique_id", "moritz_aireg_stance","moritz_aireg_pro_score","moritz_aireg_anti_score","moritz_aireg_confidence"]]
# aireg_final_mortiz_df.to_csv("aireg_final_mortiz_df.csv")
ai_final_mortiz_df = fdf[["article_set_unique_id", "moritz_ai_stance","moritz_ai_pro_score","moritz_ai_anti_score","moritz_ai_confidence"]]
ai_final_mortiz_df.to_csv("ai_final_mortiz_df.csv")

/scratch/slurm-1441438/ipykernel_368696/669698960.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fdf["moritz_ai_stance"] = labels_out            # PROREG / ANTIREG / NEUTRAL
/scratch/slurm-1441438/ipykernel_368696/669698960.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fdf["moritz_ai_pro_score"] = pro_out            # entailment prob for pro-reg
/scratch/slurm-1441438/ipykernel_368696/669698960.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try u

In [17]:
fdf[["text", "sodi_ai_stance", "moritz_ai_stance","moritz_ai_pro_score","moritz_ai_anti_score","moritz_ai_confidence"]].head()

,text,sodi_ai_stance,moritz_ai_stance,moritz_ai_pro_score,moritz_ai_anti_score,moritz_ai_confidence
0,Trump says 'male circumcision in Mozambique' i...,NEUTRAL,PESSIMISTIC,0.927706,0.998026,0.998026
1,"The historic, aging ocean liner that a Florida...",NEUTRAL,NEUTRAL,0.992974,0.999602,0.993372
2,The Associated Press is an independent global ...,NEUTRAL,NEUTRAL,0.998818,0.999160,0.999659
3,How Trump's foreign policy is reshaping the wo...,NEUTRAL,NEUTRAL,0.998296,0.999708,0.998588
4,Why the true water footprint of AI is so elusi...,UNDEFINED,NEUTRAL,0.994624,0.998937,0.995687


In [29]:
aireg_final_mortiz_df = pd.read_csv('aireg_final_mortiz_df.csv', index_col=0)
ai_final_mortiz_df = pd.read_csv('ai_final_mortiz_df.csv', index_col=0)
imm_final_mortiz_df = pd.read_csv('imm_final_mortiz_df.csv', index_col=0)
poli_final_mortiz_df = pd.read_csv('poli_final_mortiz_df.csv', index_col=0)

dfs = [aireg_final_mortiz_df, ai_final_mortiz_df, imm_final_mortiz_df, poli_final_mortiz_df]
concated = reduce(lambda left, right: pd.merge(left, right, on='article_set_unique_id', how='outer'), dfs)


In [33]:
concated.head()
concated.to_csv("wing_mortiz_scores.csv")